In [1]:
from bigmodule import M

# <aistudiograph>

# @param(id="m13", name="initialize")
# 交易引擎：初始化函数, 只执行一次
def m13_initialize_bigquant_run(context):
    import math
    import numpy as np

    from bigtrader.finance.commission import PerOrder

    # 系统已经设置了默认的交易手续费和滑点, 要修改手续费可使用如下函数
    context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))
    # 预测数据, 通过 options 传入进来, 使用 read_df 函数, 加载到内存 (DataFrame)
    # 设置买入的股票数量, 这里买入预测股票列表排名靠前的5只
    stock_count = 1
    # 每只的股票的权重, 如下的权重分配会使得靠前的股票分配多一点的资金, [0.339160, 0.213986, 0.169580, ..]
    context.stock_weights = np.array(
        [1 / math.log(i + 2) for i in range(0, stock_count)]
    )
    context.stock_weights = context.stock_weights / context.stock_weights.sum()

    # 设置每只股票占用的最大资金比例
    context.max_cash_per_instrument = 1
    context.options["hold_days"] = 1


# @param(id="m13", name="before_trading_start")
# 交易引擎：每个单位时间开盘前调用一次。
def m13_before_trading_start_bigquant_run(context, data):
    # 盘前处理，订阅行情等
    pass

# @param(id="m13", name="handle_tick")
# 交易引擎：tick数据处理函数，每个tick执行一次
def m13_handle_tick_bigquant_run(context, tick):
    pass

# @param(id="m13", name="handle_data")
# 回测引擎：每日数据处理函数, 每天执行一次
def m13_handle_data_bigquant_run(context, data):
    # 按日期过滤得到今日的预测数据
    ranker_prediction = context.data[
        context.data.date == data.current_dt.strftime("%Y-%m-%d")
    ]

    # 1. 资金分配
    # 平均持仓时间是hold_days, 每日都将买入股票, 每日预期使用 1/hold_days 的资金
    # 实际操作中, 会存在一定的买入误差, 所以在前hold_days天, 等量使用资金；之后, 尽量使用剩余资金（这里设置最多用等量的1.5倍）
    is_staging = (
        context.trading_day_index < context.options["hold_days"]
    )  # 是否在建仓期间（前 hold_days 天）
    cash_avg = context.portfolio.portfolio_value / context.options["hold_days"]
    cash_for_buy = min(context.portfolio.cash, (1 if is_staging else 1.5) * cash_avg)
    cash_for_sell = cash_avg - (context.portfolio.cash - cash_for_buy)
    positions = {
        e: p.amount * p.last_sale_price for e, p in context.portfolio.positions.items()
    }

    # 2. 生成卖出订单：hold_days天之后才开始卖出；对持仓的股票, 按机器学习算法预测的排序末位淘汰
    if not is_staging and cash_for_sell > 0:
        equities = {e: e for e, p in context.portfolio.positions.items()}
        instruments = list(
            reversed(
                list(
                    ranker_prediction.instrument[
                        ranker_prediction.instrument.apply(lambda x: x in equities)
                    ]
                )
            )
        )

        for instrument in instruments:
            context.order_target(instrument, 0)
            cash_for_sell -= positions[instrument]
            if cash_for_sell <= 0:
                break

    # 3. 生成买入订单：按机器学习算法预测的排序, 买入前面的stock_count只股票
    buy_cash_weights = context.stock_weights
    buy_instruments = list(ranker_prediction.instrument[: len(buy_cash_weights)])
    max_cash_per_instrument = (
        context.portfolio.portfolio_value * context.max_cash_per_instrument
    )
    for i, instrument in enumerate(buy_instruments):
        cash = cash_for_buy * buy_cash_weights[i]
        if cash > max_cash_per_instrument - positions.get(instrument, 0):
            # 确保股票持仓量不会超过每次股票最大的占用资金量
            cash = max_cash_per_instrument - positions.get(instrument, 0)
        if cash > 0:
            context.order_value(instrument, cash)


# @param(id="m13", name="handle_trade")
# 交易引擎：成交回报处理函数，每个成交发生时执行一次
def m13_handle_trade_bigquant_run(context, trade):
    pass

# @param(id="m13", name="handle_order")
# 交易引擎：委托回报处理函数，每个委托变化时执行一次
def m13_handle_order_bigquant_run(context, order):
    pass

# @param(id="m13", name="after_trading")
# 交易引擎：盘后处理函数，每日盘后执行一次
def m13_after_trading_bigquant_run(context, data):
    pass

# @module(position="-209,-892", comment="""因子特征，用表达式构建因子""", comment_collapsed=True)
m8 = M.input_features_dai.v30(
    mode="""表达式""",
    expr="""-- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
-- 数据&字段: 数据文档 https://bigquant.com/data/home / cn_stock_prefactors https://bigquant.com/data/datasources/cn_stock_prefactors
-- 数据使用: 表名.字段名, 对于没有指定表名的列，会从 expr_tables 推断

c_pct_rank(m_lag(amount, 1)) AS rank_amount_1
c_pct_rank(m_lag(amount, 2)) AS rank_amount_2
c_pct_rank(m_lag(close, 1)) AS rank_close_1
c_pct_rank(m_lag(close, 2)) AS rank_close_2
turn * 100 AS turn_0
m_lag(turn, 1) * 100 AS turn_1
m_lag(turn, 2) * 100 AS turn_2
m_lag(turn, 3) * 100 AS turn_3
m_lag(turn, 5) * 100 AS turn_5
close / m_lag(close, 2) AS return_1
close / m_lag(close, 3) AS return_2
close / m_lag(close, 4) AS return_3
close / m_lag(close, 6) AS return_5
close / m_lag(close, 11) AS return_10
pe_ttm AS pe_ttm_0
c_pct_rank(turn * 100) AS rank_turn_0
m_lag(daily_return, 1) * 100 AS daily_return_1
m_lag(daily_return, 2) * 100 AS daily_return_2
m_lag(daily_return, 3) * 100 AS daily_return_3
daily_return * 100 AS daily_return_0
c_pct_rank(daily_return * 100) AS rank_daily_return_0
c_pct_rank(m_lag(daily_return, 1) * 100) AS rank_daily_return_1
c_pct_rank(m_lag(daily_return, 2) * 100) AS rank_daily_return_2
c_pct_rank(m_lag(daily_return, 3) * 100) AS rank_daily_return_3
c_pct_rank(m_lag(turn, 1) * 100) AS rank_turn_1
c_pct_rank(amount) AS rank_amount_0

((close / m_lag(close, 6)) / (close / m_lag(close, 11))) AS rank_return_5_chu_rank_return_10
((close / m_lag(close, 11)) / (close / m_lag(close, 21))) AS rank_return_10_chu_rank_return_20
((close / m_lag(close, 3)) / (close / m_lag(close, 2))) AS rank_return_1_chu_rank_return_2
-- c_pct_rank(m_lag(line_price_limit=1, 1)) AS rank_line_price_limit_1
-- IF(price_limit_status = 2,1,0) AS _zt
-- IF(line_price_limit = 2,1,0) AS _zt
-- If(m_sum(_zt,2) = 1,1,0) AS _firstzt

-- open/m_lead(close,-1)-1 AS _jump

-- If(_jump > 0.01,1,0) AS _jumphigh

-- close/open-1 AS _positive

-- IF(_positive > 0.01,1,0) AS _bigpositive

-- IF( _zt = 1 and _jumphigh = 1 and _bigpositive =  1 and  _firstzt = 1,1,0) AS judge

-- IF(judge = 1, m_lead(close,1)/close-1, 0) AS return1

-- IF(judge = 1, m_lead(high,1)/close-1, 0) AS returnhigh

-- IF(return1 < 0 and returnhigh > 0,1,0) AS tr

-- IF(return1 > 0,1,0) AS po

-- IF(returnhigh < 0,1,0) AS pp

-- IF(m_lead(open,1) < close and return1 > 0,1,0) AS 低开正收

-- IF(m_lead(open,1) < close and return1 < 0,1,0) AS 低开负收

-- IF(m_lead(open,1) > close, 1,0)  AS 正开""",
    expr_filters="""-- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
-- 数据&字段: 数据文档 https://bigquant.com/data/home
-- c_pct_rank(short_return) BETWEEN 0.4 AND 0.6
-- rank_returns <= 10
-- list_days > 260
-- st_status = 0
-- m_avg(turn, 5) BETWEEN 0.02 AND 0.05
list_days > 260
st_status = 0
-- 非停牌股
suspended = 0
-- 不属于北交所
list_sector = 2
line_price_limit < 2
-- m_lag(low, 2) < m_lag(low, 1)
-- m_lag(low, 2) < m_lag(low, 1)
-- m_lag(high, 2) > m_lag(high, 1)


""",
    expr_tables="""cn_stock_prefactors""",
    extra_fields="""date, instrument""",
    order_by="""date, instrument""",
    expr_drop_na=True,
    sql="""-- 使用DAI SQL获取数据, 构建因子等, 如下是一个例子作为参考
-- DAI SQL 语法: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-sql%E5%85%A5%E9%97%A8%E6%95%99%E7%A8%8B
-- 使用数据输入1/2/3里的字段: e.g. input_1.close, input_1.* EXCLUDE(date, instrument)

SELECT
    -- 在这里输入因子表达式
    -- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
    -- 数据&字段: 数据文档 https://bigquant.com/data/home

    m_lag(close, 90) / close AS return_90,
    m_lag(close, 30) / close AS return_30,
    -- 下划线开始命名的列是中间变量, 不会在最终结果输出 (e.g. _rank_return_90)
    c_pct_rank(-return_90) AS _rank_return_90,
    c_pct_rank(return_30) AS _rank_return_30,

    c_rank(volume) AS rank_volume,
    close / m_lag(close, 1) as return_0,

    -- 日期和股票代码
    date, instrument
FROM
    -- 预计算因子 cn_stock_bar1d https://bigquant.com/data/datasources/cn_stock_bar1d
    cn_stock_prefactors
    -- SQL 模式不会自动join输入数据源, 可以根据需要自由灵活的使用
    -- JOIN input_1 USING(date, instrument)
WHERE
    -- WHERE 过滤, 在窗口等计算算子之前执行
    -- 剔除ST股票
    st_status = 0
QUALIFY
    -- QUALIFY 过滤, 在窗口等计算算子之后执行, 比如 m_lag(close, 3) AS close_3, 对于 close_3 的过滤需要放到这里
    -- 去掉有空值的行
    COLUMNS(*) IS NOT NULL
    -- _rank_return_90 是窗口函数结果，需要放在 QUALIFY 里
    AND _rank_return_90 > 0.1
    AND _rank_return_30 < 0.1
-- 按日期和股票代码排序, 从小到大
ORDER BY date, instrument
""",
    extract_data=False,
    m_name="""m8"""
)

# @module(position="-401,-798", comment="""+ 数据标注""", comment_collapsed=True)
m9 = M.input_features_dai.v30(
    input_1=m8.data,
    mode="""表达式""",
    expr="""-- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
-- 数据&字段: 数据文档 https://bigquant.com/data/home / cn_stock_prefactors https://bigquant.com/data/datasources/cn_stock_prefactors
-- 数据使用: 表名.字段名, 对于没有指定表名的列，会从 expr_tables 推断

input_1.* EXCLUDE(date, instrument)
-- (m_lead(high, 4) / m_lead(low, 1))  * (m_lead(high, 3) / m_lead(low, 1)) * (m_lead(high, 2) / m_lead(low, 1)) * (m_lead(close, 2) / m_lead(open, 1)) / (m_lead(low, 3) / m_lead(high, 1)) * ((m_lead(low, 4) / m_lead(high, 1))  * (m_lead(low, 3) / m_lead(high, 1)) * (m_lead(low, 2) / m_lead(high, 1)) * (m_lead(close, 2) / m_lead(open, 1)) / (m_lead(high, 3) / m_lead(low, 1))) AS _future_return

(m_lead(high, 4) / m_lead(low, 1))  * (m_lead(high, 3) / m_lead(low, 1)) * (m_lead(high, 2) / m_lead(low, 1)) * (m_lead(close, 2) / m_lead(open, 1)) / (m_lead(low, 3) / m_lead(high, 1)) AS _future_return

aLL_quantile_cont(_future_return, 0.01) AS _future_return_1pct
all_quantile_cont(_future_return, 0.99) AS _future_return_99pct
clip(_future_return, _future_return_1pct, _future_return_99pct) AS _clipped_return
all_cbins(_clipped_return, 100) AS label
""",
    expr_filters="""-- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
-- 数据&字段: 数据文档 https://bigquant.com/data/home
-- st_status = 0

-- 从训练数据中移除第二天涨停和跌停数据
m_lead(high, 1) != m_lead(low, 1)""",
    expr_tables="""cn_stock_bar1d""",
    extra_fields="""date, instrument""",
    order_by="""date, instrument""",
    expr_drop_na=True,
    sql="""-- 使用DAI SQL获取数据，构建因子等，如下是一个例子作为参考
-- DAI SQL 语法: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-sql%E5%85%A5%E9%97%A8%E6%95%99%E7%A8%8B

SELECT

    -- 在这里输入因子表达式
    -- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
    -- 数据&字段: 数据文档 https://bigquant.com/data/home

    c_rank(volume) AS rank_volume,
    close / m_lag(close, 1) as return_0,

    -- 日期和股票代码
    date, instrument
FROM
    -- 预计算因子 cn_stock_prefactors https://bigquant.com/data/datasources/cn_stock_prefactors
    cn_stock_prefactors
    -- 如果要使用输入数据源，需要在这里join进来
    -- JOIN input_1 USING(date, instrument)
WHERE
    -- WHERE 过滤，在窗口等计算算子之前执行
    -- 剔除ST股票
    st_status = 0
QUALIFY
    -- QUALIFY 过滤，在窗口等计算算子之后执行，比如 m_lag(close, 3) AS close_3，对于 close_3 的过滤需要放到这里
    -- 去掉有空值的行
    COLUMNS(*) IS NOT NULL
-- 按日期和股票代码排序，从小到大
ORDER BY date, instrument
""",
    extract_data=False,
    m_name="""m9"""
)

# @module(position="-397,-679", comment="""抽取训练数据""", comment_collapsed=True)
m14 = M.extract_data_dai.v17(
    sql=m9.data,
    start_date="""2022-01-01""",
    start_date_bound_to_trading_date=False,
    end_date="""2024-09-01""",
    end_date_bound_to_trading_date=False,
    before_start_days=90,
    debug=False,
    m_name="""m14"""
)

# @module(position="-274,-545", comment="""""", comment_collapsed=True)
m11 = M.stock_ranker_dai_train.v9(
    data=m14.data,
    learning_algorithm="""排序""",
    number_of_leaves=87,
    min_docs_per_leaf=160,
    number_of_trees=20,
    learning_rate=0.2,
    max_bins=1023,
    feature_fraction=1,
    data_row_fraction=1,
    plot_charts=True,
    ndcg_discount_base=1,
    m_name="""m11"""
)

# @module(position="-30,-799", comment="""抽取预测数据""", comment_collapsed=True)
m10 = M.extract_data_dai.v17(
    sql=m8.data,
    start_date="""2025-01-01""",
    start_date_bound_to_trading_date=True,
    end_date="""2026-03-13""",
    end_date_bound_to_trading_date=True,
    before_start_days=90,
    debug=False,
    m_name="""m10"""
)

# @module(position="-194,-458", comment="""""", comment_collapsed=True)
m12 = M.stock_ranker_dai_predict.v12(
    model=m11.model,
    data=m10.data,
    m_name="""m12"""
)

# @module(position="-48,-332", comment="""""", comment_collapsed=True)
m13 = M.bigtrader.v30(
    data=m12.predictions,
    start_date="""""",
    end_date="""""",
    initialize=m13_initialize_bigquant_run,
    before_trading_start=m13_before_trading_start_bigquant_run,
    handle_tick=m13_handle_tick_bigquant_run,
    handle_data=m13_handle_data_bigquant_run,
    handle_trade=m13_handle_trade_bigquant_run,
    handle_order=m13_handle_order_bigquant_run,
    after_trading=m13_after_trading_bigquant_run,
    capital_base=1000000,
    frequency="""daily""",
    product_type="""股票""",
    rebalance_period_type="""交易日""",
    rebalance_period_days="""1""",
    rebalance_period_roll_forward=True,
    backtest_engine_mode="""标准模式""",
    before_start_days=0,
    volume_limit=1,
    order_price_field_buy="""open""",
    order_price_field_sell="""close""",
    benchmark="""沪深300指数""",
    plot_charts=True,
    debug=False,
    backtest_only=False,
    m_name="""m13"""
)

# @module(position="-396,-358", comment="""""", comment_collapsed=True)
m1 = M.write_to_datasource.v14(
    input_ds=m12.predictions,
    write_to_datasource_id_="""tianzhu6_f100_160_y87""",
    primary_keys="""date,instrument""",
    partitioning="""""",
    write_mode="""追加""",
    publish_data=True,
    category="""/用户分享数据/其他""",
    price="""299/月""",
    group="""所有""",
    add_group_url='https://bigquant.com/data/categories/my-data',
    custom_price="""2000""",
    data_manage_url='https://bigquant.com/data/categories/my-data',
    raise_when_no_data=True,
    m_name="""m1"""
)
# </aistudiograph>

[2026-06-01 17:35:08] [info     ] input_features_dai.v30 开始运行 ..
[2026-06-01 17:35:09] [info     ] input_features_dai.v30 命中缓存
[2026-06-01 17:35:09] [info     ] input_features_dai.v30 运行完成 [0.881s].
[2026-06-01 17:35:09] [info     ] input_features_dai.v30 开始运行 ..
[2026-06-01 17:35:09] [info     ] input_features_dai.v30 命中缓存
[2026-06-01 17:35:09] [info     ] input_features_dai.v30 运行完成 [0.033s].
[2026-06-01 17:35:09] [info     ] extract_data_dai.v17 开始运行 ..
[2026-06-01 17:35:09] [info     ] extract_data_dai.v17 命中缓存
[2026-06-01 17:35:09] [info     ] extract_data_dai.v17 运行完成 [0.038s].
[2026-06-01 17:35:09] [info     ] stock_ranker_dai_train.v9 开始运行 ..
[2026-06-01 17:35:09] [info     ] stock_ranker_dai_train.v9 命中缓存


[2026-06-01 17:35:09] [info     ] stock_ranker_dai_train.v9 运行完成 [0.496s].
[2026-06-01 17:35:09] [info     ] extract_data_dai.v17 开始运行 ..
[2026-06-01 17:35:09] [info     ] extract_data_dai.v17 命中缓存
[2026-06-01 17:35:09] [info     ] extract_data_dai.v17 运行完成 [0.042s].
[2026-06-01 17:35:10] [info     ] stock_ranker_dai_predict.v12 开始运行 ..
[2026-06-01 17:35:10] [info     ] data: (373297, 31)
[2026-06-01 17:35:13] [info     ] stock_ranker_dai_predict.v12 运行完成 [3.864s].
[2026-06-01 17:35:14] [info     ] bigtrader.v30 开始运行 ..


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7fe6cbe51010>>
Traceback (most recent call last):
  File "/opt/pyenv/versions/3.11.8/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 770, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


[2026-06-01 17:35:14] [info     ] got metadata extra from input datasource
[2026-06-01 17:35:14] [info     ] read input 'data' ..
[2026-06-01 17:35:14] [info     ] 2025-01-01, 2026-03-13, , equity, instruments=1347
[2026-06-01 17:35:14] [info     ] bigtrader module V2.1.0
[2026-06-01 17:35:14] [info     ] bigtrader engine v0.1.0.post9+g6d7300d 2026-02-10
[2026-06-01 17:35:18] [info     ] backtest done, raw_perf_ds:dai.DataSource("_c1bdaf6f60914686bbec8c67b24697dd")


[2026-06-01 17:35:18] [info     ] bigtrader.v30 运行完成 [4.876s].
[2026-06-01 17:35:18] [info     ] write_to_datasource.v14 开始运行 ..
[2026-06-01 17:35:18] [info     ] to update data: (373297, 4), 287 dates from 2025-01-02 00:00:00 to 2026-03-13 00:00:00, 1347 instruments


Exception: handle tianzhu6_f100_160_y87/0 failed: Error creating dataset. Could not read schema from '/var/app/data/dai/datasource/user/t/ia/tianzhu6_f100_160_y87/0.feather'. Is this a 'ipc' file?: 没有写 "tianzhu6_f100_160_y87" 的权限

In [2]:
from bigmodule import M

# <aistudiograph>

# @param(id="m1", name="initialize")
# 交易引擎：初始化函数, 只执行一次
def m1_initialize_bigquant_run(context):
    import math
    import numpy as np

    from bigtrader.finance.commission import PerOrder

    # 系统已经设置了默认的交易手续费和滑点, 要修改手续费可使用如下函数
    context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))
    # 预测数据, 通过 options 传入进来, 使用 read_df 函数, 加载到内存 (DataFrame)
    # 设置买入的股票数量, 这里买入预测股票列表排名靠前的5只
    stock_count = 1
    # 每只的股票的权重, 如下的权重分配会使得靠前的股票分配多一点的资金, [0.339160, 0.213986, 0.169580, ..]
    context.stock_weights = np.array(
        [1 / math.log(i + 2) for i in range(0, stock_count)]
    )
    context.stock_weights = context.stock_weights / context.stock_weights.sum()

    # 设置每只股票占用的最大资金比例
    context.max_cash_per_instrument = 1
    context.options["hold_days"] = 1


# @param(id="m1", name="before_trading_start")
# 交易引擎：每个单位时间开盘前调用一次。
def m1_before_trading_start_bigquant_run(context, data):
    # 盘前处理，订阅行情等
    pass

# @param(id="m1", name="handle_tick")
# 交易引擎：tick数据处理函数，每个tick执行一次
def m1_handle_tick_bigquant_run(context, tick):
    pass

# @param(id="m1", name="handle_data")
# 回测引擎：每日数据处理函数, 每天执行一次
def m1_handle_data_bigquant_run(context, data):
    # 按日期过滤得到今日的预测数据
    ranker_prediction = context.data[
        context.data.date == data.current_dt.strftime("%Y-%m-%d")
    ]

    # 1. 资金分配
    # 平均持仓时间是hold_days, 每日都将买入股票, 每日预期使用 1/hold_days 的资金
    # 实际操作中, 会存在一定的买入误差, 所以在前hold_days天, 等量使用资金；之后, 尽量使用剩余资金（这里设置最多用等量的1.5倍）
    is_staging = (
        context.trading_day_index < context.options["hold_days"]
    )  # 是否在建仓期间（前 hold_days 天）
    cash_avg = context.portfolio.portfolio_value / context.options["hold_days"]
    cash_for_buy = min(context.portfolio.cash, (1 if is_staging else 1.5) * cash_avg)
    cash_for_sell = cash_avg - (context.portfolio.cash - cash_for_buy)
    positions = {
        e: p.amount * p.last_sale_price for e, p in context.portfolio.positions.items()
    }

    # 2. 生成卖出订单：hold_days天之后才开始卖出；对持仓的股票, 按机器学习算法预测的排序末位淘汰
    if not is_staging and cash_for_sell > 0:
        equities = {e: e for e, p in context.portfolio.positions.items()}
        instruments = list(
            reversed(
                list(
                    ranker_prediction.instrument[
                        ranker_prediction.instrument.apply(lambda x: x in equities)
                    ]
                )
            )
        )

        for instrument in instruments:
            context.order_target(instrument, 0)
            cash_for_sell -= positions[instrument]
            if cash_for_sell <= 0:
                break

    # 3. 生成买入订单：按机器学习算法预测的排序, 买入前面的stock_count只股票
    buy_cash_weights = context.stock_weights
    buy_instruments = list(ranker_prediction.instrument[: len(buy_cash_weights)])
    max_cash_per_instrument = (
        context.portfolio.portfolio_value * context.max_cash_per_instrument
    )
    for i, instrument in enumerate(buy_instruments):
        cash = cash_for_buy * buy_cash_weights[i]
        if cash > max_cash_per_instrument - positions.get(instrument, 0):
            # 确保股票持仓量不会超过每次股票最大的占用资金量
            cash = max_cash_per_instrument - positions.get(instrument, 0)
        if cash > 0:
            context.order_value(instrument, cash)

# @param(id="m1", name="handle_trade")
# 交易引擎：成交回报处理函数，每个成交发生时执行一次
def m1_handle_trade_bigquant_run(context, trade):
    pass

# @param(id="m1", name="handle_order")
# 交易引擎：委托回报处理函数，每个委托变化时执行一次
def m1_handle_order_bigquant_run(context, order):
    pass

# @param(id="m1", name="after_trading")
# 交易引擎：盘后处理函数，每日盘后执行一次
def m1_after_trading_bigquant_run(context, data):
    pass

# @module(position="-357,-563", comment="""""", comment_collapsed=True)
m2 = M.input_features_dai.v30(
    mode="""表达式""",
    expr="""score
position
-- input_2.close / input_1.close
""",
    expr_filters="""-- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
-- 数据&字段: 数据文档 https://bigquant.com/data/home
-- 表达式模式的过滤都是放在 QUALIFY 里, 即数据查询、计算, 最后才到过滤条件

-- c_pct_rank(-return_90) <= 0.3
-- c_pct_rank(return_30) <= 0.3
-- cn_stock_bar1d.turn > 0.02
""",
    expr_tables="""tianzhu6_f100_160_y87""",
    extra_fields="""date, instrument""",
    order_by="""date, instrument""",
    expr_drop_na=True,
    sql="""-- 使用DAI SQL获取数据, 构建因子等, 如下是一个例子作为参考
-- DAI SQL 语法: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-sql%E5%85%A5%E9%97%A8%E6%95%99%E7%A8%8B
-- 使用数据输入1/2/3里的字段: e.g. input_1.close, input_1.* EXCLUDE(date, instrument)

SELECT
    -- 在这里输入因子表达式
    -- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
    -- 数据&字段: 数据文档 https://bigquant.com/data/home

    m_lag(close, 90) / close AS return_90,
    m_lag(close, 30) / close AS return_30,
    -- 下划线开始命名的列是中间变量, 不会在最终结果输出 (e.g. _rank_return_90)
    c_pct_rank(-return_90) AS _rank_return_90,
    c_pct_rank(return_30) AS _rank_return_30,

    c_rank(volume) AS rank_volume,
    close / m_lag(close, 1) as return_0,

    -- 日期和股票代码
    date, instrument
FROM
    -- 预计算因子 cn_stock_bar1d https://bigquant.com/data/datasources/cn_stock_bar1d
    cn_stock_prefactors
    -- SQL 模式不会自动join输入数据源, 可以根据需要自由灵活的使用
    -- JOIN input_1 USING(date, instrument)
WHERE
    -- WHERE 过滤, 在窗口等计算算子之前执行
    -- 剔除ST股票
    st_status = 0
QUALIFY
    -- QUALIFY 过滤, 在窗口等计算算子之后执行, 比如 m_lag(close, 3) AS close_3, 对于 close_3 的过滤需要放到这里
    -- 去掉有空值的行
    COLUMNS(*) IS NOT NULL
    -- _rank_return_90 是窗口函数结果，需要放在 QUALIFY 里
    AND _rank_return_90 > 0.1
    AND _rank_return_30 < 0.1
-- 按日期和股票代码排序, 从小到大
ORDER BY date, instrument
""",
    extract_data=False,
    m_name="""m2"""
)

# @module(position="-356,-431", comment="""抽取预测数据""", comment_collapsed=True)
m3 = M.extract_data_dai.v17(
    sql=m2.data,
    start_date="""2025-01-01""",
    start_date_bound_to_trading_date=True,
    end_date="""2026-03-13""",
    end_date_bound_to_trading_date=True,
    before_start_days=90,
    debug=False,
    m_name="""m3"""
)

# @module(position="-321,-337", comment="""""", comment_collapsed=True)
m4 = M.data_sort.v6(
    input_ds=m3.data,
    sort_by="""position""",
    group_by="""date""",
    keep_columns="""--""",
    ascending=True,
    m_name="""m4"""
)

# @module(position="-314,-255", comment="""""", comment_collapsed=True)
m1 = M.bigtrader.v30(
    data=m4.sorted_data,
    start_date="""""",
    end_date="""""",
    initialize=m1_initialize_bigquant_run,
    before_trading_start=m1_before_trading_start_bigquant_run,
    handle_tick=m1_handle_tick_bigquant_run,
    handle_data=m1_handle_data_bigquant_run,
    handle_trade=m1_handle_trade_bigquant_run,
    handle_order=m1_handle_order_bigquant_run,
    after_trading=m1_after_trading_bigquant_run,
    capital_base=1000000,
    frequency="""daily""",
    product_type="""股票""",
    rebalance_period_type="""交易日""",
    rebalance_period_days="""1""",
    rebalance_period_roll_forward=True,
    backtest_engine_mode="""标准模式""",
    before_start_days=0,
    volume_limit=1,
    order_price_field_buy="""open""",
    order_price_field_sell="""close""",
    benchmark="""沪深300指数""",
    plot_charts=True,
    debug=False,
    backtest_only=False,
    m_name="""m1"""
)
# </aistudiograph>

[2026-03-16 17:21:40] [info     ] input_features_dai.v30 开始运行 ..
[2026-03-16 17:21:40] [info     ] expr mode
[2026-03-16 17:21:40] [info     ] input_features_dai.v30 运行完成 [0.032s].
[2026-03-16 17:21:40] [info     ] extract_data_dai.v17 开始运行 ..
[2026-03-16 17:21:40] [warning  ] start_date='2025-01-01', end_date='2026-03-13', query_start_date='2024-10-03 00:00:00' (支持加速 [url="command:switch-quota"]升级资源[/url]) ..
[2026-03-16 17:21:40] [info     ] data extracted: (373297, 4)
[2026-03-16 17:21:40] [info     ] extract_data_dai.v17 运行完成 [0.469s].
[2026-03-16 17:21:40] [info     ] data_sort.v6 开始运行 ..
[2026-03-16 17:21:41] [info     ] data_sort.v6 运行完成 [0.448s].
[2026-03-16 17:21:41] [info     ] bigtrader.v30 开始运行 ..
[2026-03-16 17:21:41] [info     ] read input 'data' ..
[2026-03-16 17:21:41] [info     ] 2025-01-02, 2026-03-13, , equity, instruments=1347
[2026-03-16 17:21:41] [info     ] bigtrader module V2.1.0
[2026-03-16 17:21:41] [info     ] bigtrader engine v0.1.0.post9+g6d7300d 2026-02-10

[2026-03-16 17:21:44] [info     ] bigtrader.v30 运行完成 [3.559s].
